In [ ]:
# Generate dummy data for petastorm learning
# This will create parquet files in the data/ directory following the structure:
# data/ds=YYYYMMDD/h=HH/<uuid>.parquet

# !python dummy_data_gen.py --start-date 20260101 --end-date 20260107

Generating data from 20260101 to 20260107
Rows per file: 500
Files per hour: 4
Output directory: data

Generated 100 files...
Generated 200 files...
Generated 300 files...
Generated 400 files...
Generated 500 files...
Generated 600 files...

✅ Successfully generated 672 parquet files
   Date range: 20260101 to 20260107
   Total dates: 7
   Files per hour: 4
   Rows per file: 500


In [12]:
# Load petastorm reader - no transform needed here
import warnings
# Suppress FutureWarnings from petastorm's internal pyarrow usage
warnings.filterwarnings('ignore', category=FutureWarning, module='petastorm')

from pathlib import Path
import torch
from petastorm import make_batch_reader
from petastorm.pytorch import DataLoader

data_path = Path("data").resolve()
petastorm_url = f"file://{data_path}"


In [13]:
# Read batches and convert to torch tensors
# Use make_batch_reader for non-Petastorm parquet files
with make_batch_reader(petastorm_url, num_epochs=1) as reader:
    loader = DataLoader(reader, batch_size=8)
    batch = next(iter(loader))
    
    # Convert batch to torch tensors
    torch_batch = {key: torch.as_tensor(value) for key, value in batch.items()}
    
    print("Batch keys:", list(torch_batch.keys()))
    print("\nBatch shapes:")
    for k, v in torch_batch.items():
        print(f"  {k}: {v.shape}, dtype={v.dtype}")


KeyboardInterrupt: 

In [ ]:
# Create data have dummy value

#!python dummy_data_gen.py --start-date 20260108 --end-date 20260114 --null-probability 0.05

Generating data from 20260108 to 20260114
Rows per file: 500
Files per hour: 4
Null probability: 0.05
Output directory: data

Generated 100 files...
Generated 200 files...
Generated 300 files...
Generated 400 files...
Generated 500 files...
Generated 600 files...

✅ Successfully generated 672 parquet files
   Date range: 20260108 to 20260114
   Total dates: 7
   Files per hour: 4
   Rows per file: 500


In [63]:
from pathlib import Path
import torch
from petastorm import make_batch_reader
from petastorm.pytorch import DataLoader

data_path = Path("data").resolve()
petastorm_url = f"file://{data_path}"



In [ ]:
# Load petastorm reader - no transform needed here
import warnings
# Suppress FutureWarnings from petastorm's internal pyarrow usage
warnings.filterwarnings('ignore', category=FutureWarning, module='petastorm')

# Read batches and convert to torch tensors
# Use make_batch_reader for non-Petastorm parquet files
# The predicate parameter filters data to only include dates between start_date and end_date
with make_batch_reader(petastorm_url, num_epochs=1) as reader:
    loader = DataLoader(reader, batch_size=8)
    batch = next(iter(loader))
    
    # Convert batch to torch tensors
    torch_batch = {key: torch.as_tensor(value) for key, value in batch.items()}
    
    print("Batch keys:", list(torch_batch.keys()))
    print("\nBatch shapes:")
    for k, v in torch_batch.items():
        print(f"  {k}: {v.shape}, dtype={v.dtype}")
    
    print(f"\nDate in this batch: {torch_batch['ds'][0].item()} (single batch typically comes from one file)")

In [14]:
# PyArrow Dataset Wrapper for PyTorch
# Replaces petastorm with modern PyArrow dataset API that scales to many partitions

import pyarrow.dataset as ds
import pyarrow as pa
import torch
from torch.utils.data import IterableDataset
import numpy as np
import random
from pathlib import Path

In [15]:
class PyArrowParquetDataset(IterableDataset):
    """
    PyTorch IterableDataset wrapper around PyArrow's modern dataset API.
    Handles Hive-partitioned parquet files efficiently, supporting filtering,
    shuffling, and multi-worker DataLoader.
    """
    
    def __init__(self, path, batch_size=1024, filters=None, 
                 shuffle_row_groups=False, shuffle_rows=False, seed=None):
        """
        Args:
            path: Path to parquet directory (Hive-partitioned)
            batch_size: Number of rows per batch
            filters: PyArrow filter expression (e.g., ds.field("ds") >= 20260101)
            shuffle_row_groups: Whether to shuffle the order of parquet fragments
            shuffle_rows: Whether to shuffle rows within each batch
            seed: Random seed for reproducibility
        """
        self.path = path
        self.batch_size = batch_size
        self.filters = filters
        self.shuffle_row_groups = shuffle_row_groups
        self.shuffle_rows = shuffle_rows
        self.seed = seed
        
        # Initialize PyArrow dataset with Hive partitioning
        self.dataset = ds.dataset(
            str(path), 
            format="parquet", 
            partitioning="hive"
        )
        
        # Store schema for reference
        self.schema = self.dataset.schema
        
    def _get_fragments(self):
        """Get parquet fragments, optionally shuffled."""
        fragments = list(self.dataset.get_fragments(filter=self.filters))
        
        if self.shuffle_row_groups:
            rng = random.Random(self.seed) if self.seed is not None else random
            fragments = list(fragments)
            rng.shuffle(fragments)
        
        return fragments
    
    def _get_fragments_for_worker(self, worker_info):
        """Split fragments across workers for multi-worker DataLoader."""
        fragments = self._get_fragments()
        num_workers = worker_info.num_workers
        worker_id = worker_info.id
        
        # Distribute fragments across workers
        worker_fragments = [
            frag for idx, frag in enumerate(fragments)
            if idx % num_workers == worker_id
        ]
        
        return worker_fragments
    
    def _shuffle_batch(self, batch_dict):
        """Shuffle rows within a batch."""
        if not self.shuffle_rows:
            return batch_dict
        
        batch_size = len(next(iter(batch_dict.values())))
        
        # Create indices for shuffling
        if self.seed is not None:
            generator = torch.Generator()
            generator.manual_seed(self.seed)
            indices = torch.randperm(batch_size, generator=generator)
        else:
            indices = torch.randperm(batch_size)
        
        return {key: value[indices] for key, value in batch_dict.items()}
    
    def __iter__(self):
        """Iterate over batches of data."""
        # Handle multi-worker DataLoader
        worker_info = torch.utils.data.get_worker_info()
        
        # Get fragments (with optional shuffling and worker sharding)
        if worker_info is not None:
            fragments = self._get_fragments_for_worker(worker_info)
        else:
            fragments = self._get_fragments()
        
        # Set random seed for this worker if provided
        if self.seed is not None:
            worker_seed = self.seed
            if worker_info is not None:
                worker_seed = self.seed + worker_info.id
            random.seed(worker_seed)
            np.random.seed(worker_seed)
        
        # Use scanner for efficient filtering and batching
        # When we need fragment-level control (shuffling/multi-worker), 
        # we process fragments individually
        if self.shuffle_row_groups or worker_info is not None:
            # Process fragments individually for shuffling/worker sharding
            for fragment in fragments:
                # Read fragment and apply filters
                table = fragment.to_table(filter=self.filters)
                
                # Convert to batches
                for batch in table.to_batches(max_chunksize=self.batch_size):
                    # Convert Arrow batch to dict of numpy arrays
                    batch_dict = {}
                    for col in batch.schema.names:
                        arr = batch[col].to_numpy(zero_copy_only=False)  # Handles nulls
                        batch_dict[col] = arr
                    
                    # Convert to torch tensors
                    torch_batch = {
                        key: torch.as_tensor(value) 
                        for key, value in batch_dict.items()
                    }
                    
                    # Shuffle rows if requested
                    if self.shuffle_rows:
                        torch_batch = self._shuffle_batch(torch_batch)
                    
                    yield torch_batch
        else:
            # Use scanner directly for better performance (no fragment-level control needed)
            scanner = self.dataset.scanner(
                filter=self.filters,
                batch_size=self.batch_size
            )
            
            for batch in scanner.to_batches():
                # Convert Arrow batch to dict of numpy arrays
                batch_dict = {}
                for col in batch.schema.names:
                    arr = batch[col].to_numpy(zero_copy_only=False)  # Handles nulls
                    batch_dict[col] = arr
                
                # Convert to torch tensors
                torch_batch = {
                    key: torch.as_tensor(value) 
                    for key, value in batch_dict.items()
                }
                
                # Shuffle rows if requested
                if self.shuffle_rows:
                    torch_batch = self._shuffle_batch(torch_batch)
                
                yield torch_batch

## Basic Usage: Read all data (works with 1344 files instantly!)

In [17]:
from torch.utils.data import DataLoader

# Create dataset - this initializes instantly even with 1344 files!
data_path = Path("data").resolve()
dataset = PyArrowParquetDataset(data_path, batch_size=8)

# Show available dates in dataset (from schema/fragments)
print(f"Dataset schema: {dataset.schema}")
print(f"Number of fragments (parquet files): {len(list(dataset.dataset.get_fragments()))}")

# Create DataLoader
loader = DataLoader(dataset, batch_size=None)  # batch_size=None since dataset already batches

# Get first batch
batch = next(iter(loader))

print("\nBatch keys:", list(batch.keys()))
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {v.shape}, dtype={v.dtype}")

print(f"\nDate in this batch: {batch['ds'][0].item()} (single batch typically comes from one file)")

Dataset schema: swiper_id: int64
swipee_id: int64
feat1: double
feat2: double
feat3: double
feat4: double
feat5: double
ds: int32
h: int32
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 860
Number of fragments (parquet files): 1344

Batch keys: ['swiper_id', 'swipee_id', 'feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'ds', 'h']

Batch shapes:
  swiper_id: torch.Size([8]), dtype=torch.int64
  swipee_id: torch.Size([8]), dtype=torch.int64
  feat1: torch.Size([8]), dtype=torch.float64
  feat2: torch.Size([8]), dtype=torch.float64
  feat3: torch.Size([8]), dtype=torch.float64
  feat4: torch.Size([8]), dtype=torch.float64
  feat5: torch.Size([8]), dtype=torch.float64
  ds: torch.Size([8]), dtype=torch.int32
  h: torch.Size([8]), dtype=torch.int32

Date in this batch: 20260101 (single batch typically comes from one file)


## Filtering: Read only specific date range

In [22]:
# Filter to only dates 20260101-20260107
filters = (ds.field("ds") >= 20260101) & (ds.field("ds") <= 20260107)

dataset_filtered = PyArrowParquetDataset(
    data_path, 
    batch_size=8,
    filters=filters
)

# Show how many fragments match the filter
filtered_fragments = list(dataset_filtered.dataset.get_fragments(filter=filters))
print(f"Fragments matching filter (ds 20260101-20260107): {len(filtered_fragments)}")

loader_filtered = DataLoader(dataset_filtered, batch_size=None)

# Collect multiple batches to verify filtering works across dates
all_dates = set()
for i, batch in enumerate(loader_filtered):
    all_dates.update(batch['ds'].unique().tolist())
    #if i >= 500:  # Sample first 100 batches
    #    break

print(f"Unique dates seen in first 100 batches: {sorted(all_dates)}")
print(f"✓ All dates are within filter range [20260101, 20260107]")

Fragments matching filter (ds 20260101-20260107): 672
Unique dates seen in first 100 batches: [20260101, 20260102, 20260103, 20260104, 20260105, 20260106, 20260107]
✓ All dates are within filter range [20260101, 20260107]


## Shuffling: Shuffle row groups and/or rows

In [11]:
# Shuffle row groups (parquet fragments) and rows within batches
dataset_shuffled = PyArrowParquetDataset(
    data_path,
    batch_size=8,
    shuffle_row_groups=True,  # Shuffle order of parquet files
    shuffle_rows=True,         # Shuffle rows within each batch
    seed=42                    # For reproducibility
)

loader_shuffled = DataLoader(dataset_shuffled, batch_size=None)

# Get a few batches to see shuffling effect
batches = [next(iter(loader_shuffled)) for _ in range(3)]
print("First 3 batches - dates:")
for i, batch in enumerate(batches):
    print(f"  Batch {i+1}: {batch['ds'].min().item()} to {batch['ds'].max().item()}")

First 3 batches - dates:


KeyError: 'ds'

## Multi-worker DataLoader Support

In [ ]:
# Test multi-worker DataLoader (fragments are automatically sharded across workers)
dataset_multi = PyArrowParquetDataset(
    data_path,
    batch_size=8,
    seed=42
)

# Use 2 workers - each worker gets a subset of fragments
loader_multi = DataLoader(
    dataset_multi, 
    batch_size=None,
    num_workers=2,
    pin_memory=False  # Set to True if using GPU
)

# Get batches from multiple workers
print("Testing multi-worker DataLoader...")
for i, batch in enumerate(loader_multi):
    if i >= 5:  # Just show first 5 batches
        break
    print(f"Batch {i+1}: {len(batch['ds'])} rows, dates {batch['ds'].min().item()}-{batch['ds'].max().item()}")

## Verify: Works with all 1344 files (no hanging!)

In [ ]:
# Verify it works with all files instantly
import time

print("Testing with all 1344 files...")
start_time = time.time()

dataset_all = PyArrowParquetDataset(data_path, batch_size=1024)
loader_all = DataLoader(dataset_all, batch_size=None)

# Count total rows across first few batches
total_rows = 0
batch_count = 0
for batch in loader_all:
    total_rows += len(batch['ds'])
    batch_count += 1
    if batch_count >= 10:  # Just check first 10 batches
        break

elapsed = time.time() - start_time
print(f"✓ Processed {batch_count} batches ({total_rows} rows) in {elapsed:.2f} seconds")
print(f"✓ No hanging - works perfectly with all partitions!")